## Running Conditional VGAE

- Dim-reduction on features ($x \in \mathbb{R}^{N \times P}$: observation; $z \in \mathbb{R}^{N \times D}$: representation; interpretability as "trajectory / gradient")
- Benchmark against K-means clustering & PCA, regular VAE, etc.

In [ ]:
import os
import gc
import sys
import pickle
import gzip

import numpy as np
import cupy as cp
import pandas as pd
import scanpy as sc

import torch
import tifffile
import torch.nn as nn

import networkx as nx
import tifffile
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.neighbors import NearestNeighbors
from scipy import sparse
from torch_geometric import utils as pyg_utils


In [ ]:
from ipywidgets import interact, widgets
from IPython.display import display

from matplotlib import rcParams
rcParams.update({'font.size': 10})
rcParams.update({'figure.dpi': 300})
rcParams.update({'savefig.dpi': 300})

import warnings
warnings.filterwarnings('ignore')

%matplotlib inline

In [ ]:
sys.path.append('..')
sys.path.append('../models/')
sys.path.append('../util')

import IO, utils, plot, gen_graph, configs, dataset, zonation
import baseline, sb_vgae, model_train, model_eval

from torch_geometric.loader import DataLoader

### VGAE (Xenium-DESI integration)

- Integrate DESI prior to guide latent prob. clustering of Xenium

In [ ]:
import json
import squidpy as sq

from util import IO, utils, gen_graph
from models import  dataset
from torch_geometric import utils as pyg_utils

from importlib import reload
reload(IO)
reload(utils)
reload(gen_graph)
reload(dataset)


In [ ]:
# Load paired Xenium & DESI

xenium_path = '../data/xenium/'
desi_path = '../data/integrated/desi_xenium_map/'
n_aux = 10
run_single_sample = True

if run_single_sample:
    sample_id = 'NIH_F5'
    adata = IO.load_xenium(os.path.join(xenium_path, sample_id))  # Load xenium
    adata_desi = IO.load_desi(os.path.join(desi_path, sample_id+'.h5ad'), raw_img=False)  # Load DESI
    adata, adata_desi = IO.filter_cells(adata, adata_desi)  # Filter common cells

    # Dimension reduction on aux. variable
    utils.get_PCs(adata_desi, n_pcs=n_aux)
    adata.obsm['X_aux'] = adata_desi.obsm['X_pca']

else:
    # All samples
    sample_ids = sorted([f for f in os.listdir(xenium_path) 
                         if os.path.isdir(os.path.join(xenium_path, f))])
    
    adatas = []
    n_aux = 10
    
    for sample_id in sample_ids:
        print('Loading {}...'.format(sample_id))
        adata = IO.load_xenium(os.path.join(xenium_path, sample_id))  # Load xenium
        adata_desi = IO.load_desi(os.path.join(desi_path, sample_id+'.h5ad'), raw_img=False)  # Load DESI
        adata, adata_desi = IO.filter_cells(adata, adata_desi)  # Filter common cells
    
        # Dimension reduction on aux. variable
        utils.get_PCs(adata_desi, n_pcs=n_aux)
        adata.obsm['X_aux'] = adata_desi.obsm['X_pca']
    
        adatas.append(adata)
        del adata_desi

gc.collect()


#### Model sketch iVAE: 
- $z_i \mid u_i \sim \mathcal{N}(f_{\lambda_{\mu}}(u_i), f_{\lambda_{\sigma}}(u_i))$ vs. $z_i \mid u_i \sim \mathcal{vMF}(f_{\lambda}(u_i))$
- $x_i \mid z_i, \mathbf{A} \sim \mathcal{NegBinom}(l \cdot f_{\theta}(z_i, \mathbf{A}, \theta_g)$

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import pyro

from torch_geometric.data import DataLoader
from models import vgae, model_train

In [ ]:
xenium_path = '../data/xenium/'
sample_ids = sorted([f for f in os.listdir(xenium_path) 
                         if os.path.isdir(os.path.join(xenium_path, f))])

In [ ]:
split_ratio = 0.6
female_indices = np.arange(len(sample_ids)//2)
male_indices = np.arange(len(sample_ids)//2, len(sample_ids))

train_indices = list(np.random.choice(female_indices, size=int(len(female_indices)*split_ratio), replace=False)) + \
                list(np.random.choice(male_indices, size=int(len(male_indices)*split_ratio), replace=False))


In [ ]:
if run_single_sample:
    loader = dataset.XeniumGraphDataset(k=30, n_subgraphs=10)
    graph_data = loader.load_graphs([adata])
    train_dl = DataLoader(graph_data, shuffle=True)
else:
    # Randomly split samples for training / test (per gender)
    split_ratio = 0.6
    female_indices = np.arange(len(adatas)//2)
    male_indices = np.arange(len(adatas)//2, len(adatas))
    
    train_indices = list(np.random.choice(female_indices, size=int(len(female_indices)*split_ratio), replace=False)) + \
                    list(np.random.choice(male_indices, size=int(len(male_indices)*split_ratio), replace=False))

    val_indices = [i for i in np.arange(adatas) 
                   if i not in train_indices]
    print('Validation indices': val_indices)

    loader = dataset.XeniumGraphDataset(k=30, n_subgraphs=10)
    train_data = loader.load_graphs([adatas[i] for i in train_indices])
    train_dl = DataLoader(train_data, shuffle=True)
    

In [ ]:
from importlib import reload

reload(IO)
reload(utils)
reload(plot)
reload(configs)
reload(vgae)
reload(model_train)

torch.manual_seed(42)

In [ ]:
lr = 1e-3
n_epochs = 300
device = torch.device('cuda')
n_aux = adata.obsm['X_aux'].shape[1]

train_configs = configs.set_train_configs(n_epochs=n_epochs, 
                                          lr=lr,
                                          gamma=0.5,
                                          annealing=False,
                                          device=device)

model_configs = configs.set_model_configs(device=device, prior='normal', act=nn.SiLU(),
                                          beta=0.5, c_in=adata.shape[1], c_aux=n_aux,
                                          c_hidden=8, c_latent=8,
                                          k_hop=3, dropout=0.5) 

In [ ]:
pyro.clear_param_store()
model = vgae.VGAE(model_configs, device=train_configs.device)
model, losses = model_train.train_vgae(model, train_dl, train_configs)

In [ ]:
plt.figure(figsize=(5, 2))
plt.plot(np.arange(train_configs.n_epochs), losses)
plt.xlabel('Epochs')
plt.ylabel('Train ELBO')
plt.show()

In [ ]:
torch.save(model.state_dict(), os.path.join('../results/', '{}_multi.pt').format(model_configs.prior))

**TODO:** Attach customized trajectory metadata to adata.uns['graph']['pseudotime_list'] for GAM association tests

#### Held-out validation

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import scFates as scf
import igraph

import json
import squidpy as sq

import pyro
from pyro.infer import SVI, Trace_ELBO, infer_discrete

from util import IO, utils, gen_graph, trajectory
from models import vgae, model_train, dataset
from torch_geometric import utils as pyg_utils
from torch_geometric.data import DataLoader

from importlib import reload
reload(utils)
reload(gen_graph)
reload(dataset)
reload(trajectory)

In [ ]:
def evaluate_elbo(model, dataloader, device=torch.device('cpu')):
    from pyro.optim import Adam
    optimizer = Adam({"lr": 1.0e-3})  # dummy optimizer
    
    model = model.to(device)
    elbo = Trace_ELBO()
    svi = SVI(model.model, model.guide, optimizer, elbo)
    elbos = []
    
    for data in dataloader:
        x = data.x.to(device).float()
        u = data.u.to(device).float()
        edge_index = data.edge_index.to(device)
        elbo = svi.evaluate_loss(x, u, edge_index)

        elbos.append(elbo / x.shape[0])

    return elbos

def evaluate_kl(model, dataloader, device=torch.device('cpu')):
    from pyro.optim import Adam
    from pyro.infer import TraceMeanField_ELBO
    from pyro import poutine
    
    optimizer = Adam({"lr": 1.0e-3})  # dummy optimizer
    
    model = model.to(device)
    elbo = TraceMeanField_ELBO()
    svi = SVI(model.model, model.guide, optimizer, elbo)
    
    kl_divs = []
    for data in dataloader:
        x = data.x.to(device).float()
        u = data.u.to(device).float()
        edge_index = data.edge_index.to(device)

        with poutine.scale(scale=1e-7):
            kl_div = svi.evaluate_loss(x, u, edge_index)
            kl_divs.append(kl_div)

    return kl_divs

def cell_type_dynamics(
    adata,
    cell_type, 
    show_fig=False
):
    """
    Dynamics of cell-type proportion along discrete zonations
    """
    assert 't_discrete' in adata.obs.columns and 'cell_type' in adata.obs.columns
    assert cell_type in adata.obs['cell_type'].unique()

    zonation_states = np.unique(adata.obs['t_discrete'])
    dynamics = []
    for state in zonation_states:
        summary = adata[adata.obs['t_discrete'] == state].obs['cell_type'].value_counts()
        if cell_type in summary.index:
            target_count = summary[cell_type]
            total_count = summary.sum()
            dynamics.append(target_count / total_count)
        else:
            dynamics.append(0)

    if show_fig:
        fig, ax = plt.subplots(figsize=(4, 1))
        ax.plot(zonation_states, dynamics, '.--', c='blue')
        ax.set_title(cell_type)
        plt.show()
    
    return dynamics

In [ ]:
# Load validation data
xenium_path = '../data/xenium/'
desi_path = '../data/integrated/desi_xenium_map/'
n_aux = 10
n_latent = 8

# sample_ids = sorted([f for f in os.listdir(xenium_path) 
#                          if os.path.isdir(os.path.join(xenium_path, f))])
# train_ids = sample_ids[train_indices]
# val_ids = sample_ids[val_indices]


train_ids = ['NIH_F1', 'NIH_F2', 'NIH_F3', 'NIH_M1', 'NIH_M2', 'NIH_M3']
val_ids = ['NIH_F4', 'NIH_F5', 'NIH_M4', 'NIH_M5']

adatas_train = []
for sample_id in train_ids:
    print('Loading training sample {}...'.format(sample_id))
    adata = IO.load_xenium(os.path.join(xenium_path, sample_id))
    adata_desi = IO.load_desi(os.path.join(desi_path, sample_id+'.h5ad'), raw_img=False)
    adata, adata_desi = IO.filter_cells(adata, adata_desi)
    
    utils.get_PCs(adata_desi, n_pcs=n_aux)
    adata.obsm['X_aux'] = adata_desi.obsm['X_pca'].astype(np.float32)
    
    adatas_train.append(adata)
    del adata_desi
    gc.collect()
    
adatas_val = []
for sample_id in val_ids:
    print('Loading val sample {}...'.format(sample_id))
    adata = IO.load_xenium(os.path.join(xenium_path, sample_id))
    adata_desi = IO.load_desi(os.path.join(desi_path, sample_id+'.h5ad'), raw_img=False)
    adata, adata_desi = IO.filter_cells(adata, adata_desi)
    
    utils.get_PCs(adata_desi, n_pcs=n_aux)
    adata.obsm['X_aux'] = adata_desi.obsm['X_pca'].astype(np.float32)
    
    adatas_val.append(adata)
    del adata_desi
    gc.collect()

del sample_id

In [ ]:
# Load models
prior_choice = 'normal'
device = torch.device('cuda')
model_configs = configs.set_model_configs(device=device, prior=prior_choice, act=nn.SiLU(),
                                          beta=0.5, c_in=adata.shape[1], c_aux=n_aux,
                                          c_hidden=8, c_latent=n_latent,
                                          k_hop=3, dropout=0.5) 


model_normal = vgae.VGAE(configs=model_configs, device=device)
model_normal.load_state_dict(torch.load('../results/{}_multi.pt'.format(prior_choice)))

prior_choice = 'vMF'
model_configs = configs.set_model_configs(device=device, prior=prior_choice, act=nn.SiLU(),
                                          beta=0.5, c_in=adata.shape[1], c_aux=n_aux,
                                          c_hidden=8, c_latent=n_latent,
                                          k_hop=3, dropout=0.5) 

model_vMF = vgae.VGAE(configs=model_configs, device=device)
model_vMF.load_state_dict(torch.load('../results/{}_multi.pt'.format(prior_choice)))

Fitting metrics:
- Neg. ELBO
- Reconstruction NLL
- KL div.

In [ ]:
# Create subgraph-batched graph data
loader = dataset.XeniumGraphDataset(k=30, n_subgraphs=8)
train_data = loader.load_graphs(adatas_train)
val_data = loader.load_graphs(adatas_val)
train_dl = DataLoader(train_data)
val_dl = DataLoader(val_data)

del train_data, val_data
gc.collect()

In [ ]:
train_elbos_normal = evaluate_elbo(model_normal, train_dl, device=device)
val_elbos_normal = evaluate_elbo(model_normal, val_dl, device=device)

train_elbos_vMF = evaluate_elbo(model_vMF, train_dl, device=device)
val_elbos_vMF = evaluate_elbo(model_vMF, val_dl, device=device)

gc.collect()
torch.cuda.empty_cache()

In [ ]:
elbo_df = pd.DataFrame({
    'ELBO': train_elbos_normal + train_elbos_vMF + \
            val_elbos_normal + val_elbos_vMF,
    
    'Label': ['train']*len(train_elbos_normal)*2 + \
             ['val']*len(val_elbos_normal)*2,
    
    'Prior': ['Normal']*len(train_elbos_normal) + \
             ['vMF']*len(train_elbos_vMF) + \
             ['Normal']*len(val_elbos_normal) + \
             ['vMF']*len(val_elbos_vMF)
})

fig, ax = plt.subplots(figsize=(6, 4))
ax = sns.barplot(x='Label', y='ELBO', data=elbo_df, hue='Prior')
ax.spines[['right', 'top']].set_visible(False)
sns.move_legend(ax, "upper right", bbox_to_anchor=(1.2, 0.7))
ax.set_title('ELBO')
plt.show()

In [ ]:
train_kls_normal = evaluate_kl(model_normal, train_dl, device=device)
val_kls_normal = evaluate_kl(model_normal, val_dl, device=device)

train_kls_vMF = evaluate_kl(model_vMF, train_dl, device=device)
val_kls_vMF = evaluate_kl(model_vMF, val_dl, device=device)

gc.collect()
torch.cuda.empty_cache()

In [ ]:
kl_df = pd.DataFrame({
    'KL': train_kls_normal + train_kls_vMF + \
            val_kls_normal + val_kls_vMF,
    
    'Label': ['train']*len(train_kls_normal)*2 + \
             ['val']*len(val_kls_normal)*2,
    
    'Prior': ['Normal']*len(train_kls_normal) + \
             ['vMF']*len(train_kls_vMF) + \
             ['Normal']*len(val_kls_normal) + \
             ['vMF']*len(val_kls_vMF)
})

fig, ax = plt.subplots(figsize=(6, 4))
ax = sns.barplot(x='Label', y='KL', data=kl_df, hue='Prior')
ax.spines[['right', 'top']].set_visible(False)
sns.move_legend(ax, "upper right", bbox_to_anchor=(1.2, 0.7))
ax.set_title('KL divergence')
plt.show()

In [ ]:
train_r2s_normal = []
train_r2s_vMF = []

for i, adata in enumerate(adatas_train):
    print('Evaluating on train sample {}...'.format(i))
    coords = adata.obs[['y_centroid', 'x_centroid']].copy().to_numpy() 
    G = gen_graph.construct_graph(coords, 
                                  k=30, 
                                  r=np.inf)
    edge_index = pyg_utils.from_networkx(G).edge_index
    del G
    
    model_normal.device = 'cpu'
    model_normal = model_normal.to('cpu')
    pred_params = model_normal.predict(adata.X.A, 
                                       adata.obsm['X_aux'], 
                                       edge_index)
    px = pred_params.px.detach().cpu().numpy()

    _, _, r_val, _, _ = linregress(adata.X.A.flatten(), px.flatten())
    train_r2s_normal.append(r_val**2)
    
    model_vMF.device = 'cpu'
    model_vMF = model_vMF.to('cpu')
    pred_params = model_vMF.predict(adata.X.A, 
                                    adata.obsm['X_aux'], 
                                    edge_index)
    px = pred_params.px.detach().cpu().numpy()

    _, _, r_val, _, _ = linregress(adata.X.A.flatten(), px.flatten())
    train_r2s_vMF.append(r_val**2)

    del edge_index, adata
    gc.collect()
    torch.cuda.empty_cache()

In [ ]:
val_r2s_normal = []
val_r2s_vMF = []

for i, adata in enumerate(adatas_val):
    print('Evaluating on val sample {}...'.format(i))
    coords = adata.obs[['y_centroid', 'x_centroid']].copy().to_numpy() 
    G = gen_graph.construct_graph(coords, 
                                  k=30, 
                                  r=np.inf)
    edge_index = pyg_utils.from_networkx(G).edge_index
    del G
    
    model_normal.device = 'cpu'
    model_normal = model_normal.to('cpu')
    pred_params = model_normal.predict(adata.X.A, 
                                       adata.obsm['X_aux'], 
                                       edge_index)
    px = pred_params.px.detach().cpu().numpy()

    _, _, r_val, _, _ = linregress(adata.X.A.flatten(), px.flatten())
    val_r2s_normal.append(r_val**2)
    
    model_vMF.device = 'cpu'
    model_vMF = model_vMF.to('cpu')
    pred_params = model_vMF.predict(adata.X.A, 
                                    adata.obsm['X_aux'], 
                                    edge_index)
    px = pred_params.px.detach().cpu().numpy()

    _, _, r_val, _, _ = linregress(adata.X.A.flatten(), px.flatten())
    val_r2s_vMF.append(r_val**2)

    del edge_index, adata
    gc.collect()
    torch.cuda.empty_cache()

In [ ]:
recon_df = pd.DataFrame({
    r'$R^2$': train_r2s_normal + train_r2s_vMF + \
             val_r2s_normal + val_r2s_vMF,
    
    'Label': ['train']*len(train_r2s_normal)*2 + \
             ['val']*len(val_r2s_normal)*2,
    
    'Prior': ['Normal']*len(train_r2s_normal) + \
             ['vMF']*len(train_r2s_vMF) + \
             ['Normal']*len(val_r2s_normal) + \
             ['vMF']*len(val_r2s_vMF)
})

fig, ax = plt.subplots(figsize=(6, 4))
ax = sns.barplot(x='Label', y=r'$R^2$', data=recon_df, hue='Prior')
ax.spines[['right', 'top']].set_visible(False)
sns.move_legend(ax, "upper right", bbox_to_anchor=(1.2, 0.7))
ax.set_title('Reconstruction fitting')
plt.show()

In [ ]:
# Visualize disentanglement btw latent features

# sns.clustermap(np.corrcoef(qz.T), cmap='coolwarm')
# plot.disp_spatial_latents(adata, qz, ncols=4)

In [ ]:
# rand_indices = np.random.choice(np.arange(adatas_val[-1].X.A.shape[1]), size=100, replace=False)

# plt.figure(figsize=(6, 4))
# plt.scatter(adatas_val[-1].X.A[:, rand_indices].flatten(), px[:, rand_indices].flatten(), s=1)
# plt.xlabel('X')
# plt.ylabel("X_reconst")
# plt.show()

# del rand_indices

Trajectory metrics:
- Spearman(gradients, antibody)

In [ ]:
from sklearn.metrics import normalized_mutual_info_score
from scipy.stats import pearsonr, spearmanr

In [ ]:
def load_ab_stain(filename, adata_ref):
    n_cells = adata_ref.shape[0]

    # load raw images, skip DAPI channel
    img = tifffile.imread(filename)[1:]  # skip DAPI

    # Filter indices mapped to reference `adata`
    coords = np.round(
        adata_ref.obs[['y_centroid', 'x_centroid']].copy().to_numpy().T
    ).astype(np.int16)  # dim: [Y*X, 2], YX-index 

    adata = sc.AnnData(
        np.array([chan[tuple(coords)] for chan in img]).T
    )
    adata.obs['y_centroid'], adata.obs['x_centroid'] = coords
    adata.obsm['spatial'] = np.array([coords[1], coords[0]]).T  # XY-index
    IO.load_spatial_metadata(adata, load_img=False)

    try:
        labels = IO.load_ome_labels(filename)[1:]
        adata.var_names = labels
    except ET.ParseError:
        pass

    return adata

In [ ]:
n_latent = 8
display_trajectory = True
spearmans_normal = []
spearmans_vMF = []

for i, sample_id in enumerate(val_ids):
    # ----------------------------------------
    print('Evaluating against Ab-staining on {}...'.format(sample_id))
    print('\t Computing latent representation...')
    
    adata = adatas_val[i]
    n_cells = adata.shape[0]
    coords = adata.obs[['y_centroid', 'x_centroid']].copy().to_numpy() 
    G = gen_graph.construct_graph(coords, 
                                  k=30, 
                                  r=np.inf)
    edge_index = pyg_utils.from_networkx(G).edge_index
    del G
    
    model_normal.device = 'cpu'
    model_normal = model_normal.to('cpu')
    pred_params = model_normal.predict(adata.X.A, 
                                       adata.obsm['X_aux'], 
                                       edge_index)
    qz_normal = pred_params.qz_params[0].detach().cpu().numpy()
    
    model_vMF.device = 'cpu'
    model_vMF = model_vMF.to('cpu')
    pred_params = model_vMF.predict(adata.X.A, 
                                    adata.obsm['X_aux'], 
                                    edge_index)
    qz_vMF = pred_params.qz_params.detach().cpu().numpy()
    
    del edge_index
    gc.collect()

    
    # ----------------------------------------
    print('\t Computing gradients / trajectory...')
    adata_qz_normal = sc.AnnData(qz_normal)
    scf.tl.curve(adata_qz_normal, 
             Nodes=n_latent, 
             epg_extend_leaves=True,
             ndims_rep=n_latent)

    trajectory.compute_trajectory(adata_qz_normal, 
                                  dist_metric='normal', 
                                  k=0)

    if display_trajectory:
        sc.pp.pca(adata_qz_normal)
        sc.pp.neighbors(adata_qz_normal)
        sc.tl.umap(adata_qz_normal, n_components=3)
    
        adata.obs['t'] = adata_qz_normal.obs['t'].values
        adata.obs['t_discrete'] = adata_qz_normal.obs['t_discrete'].values
        
        scf.pl.graph(adata_qz_normal, basis='umap', nodes=list(np.arange(n_latent)))        
        sq.pl.spatial_scatter(adata, color='t', 
                              cmap='RdBu_r', size=20, img=False,
                              title='Pseudotime\n (Normal Prior)')
        
        sq.pl.spatial_scatter(adata, color='t_discrete', 
                              cmap='tab20', size=20, img=False,
                              title='Zonation\n (Normal Prior)')
    
    adata_qz_vMF = sc.AnnData(qz_vMF)
    scf.tl.curve(adata_qz_vMF, 
                 Nodes=n_latent, 
                 epg_extend_leaves=True,
                 ndims_rep=n_latent)

    trajectory.compute_trajectory(adata_qz_vMF, 
                                  dist_metric='vMF', 
                                  k=0)

    if display_trajectory:
        sc.pp.pca(adata_qz_vMF)
        sc.pp.neighbors(adata_qz_vMF)
        sc.tl.umap(adata_qz_vMF, n_components=3)
        
        adata.obs['t'] = adata_qz_vMF.obs['t'].values
        adata.obs['t_discrete'] = adata_qz_vMF.obs['t_discrete'].values
        
        scf.pl.graph(adata_qz_vMF, basis='umap', nodes=list(np.arange(n_latent)))
        sq.pl.spatial_scatter(adata, color='t', 
                              cmap='RdBu_r', size=20, img=False,
                              title='Pseudotime\n (vMF Prior)')
        
        sq.pl.spatial_scatter(adata, color='t_discrete', 
                              cmap='tab20', size=20, img=False,
                              title='Zonation\n (vMF Prior)')

    
    # ----------------------------------------
    print('\t Loading paired Ab staining expressions...')
    filename = '../data/integrated/antibody/{}.ome.tif'.format(sample_id)
    adata_ab = IO.load_ab_stain(filename, adata)

    spearman_norm = [
        spearmanr(adata_qz_normal.obs['t'], adata_ab[:, label].X.squeeze())[0]
        for label in adata_ab.var_names
    ]
    spearmans_normal.append(spearman_norm)

    spearman_vMF = [
        spearmanr(adata_qz_vMF.obs['t'], adata_ab[:, label].X.squeeze())[0]
        for label in adata_ab.var_names
    ]
    spearmans_vMF.append(spearman_vMF)

    print('====================================\n\n')
    del adata, adata_qz_normal, adata_qz_vMF, adata_ab
    gc.collect()


In [ ]:
spearmans_normal = np.array(spearmans_normal)
spearmans_vMF = np.array(spearmans_vMF)

In [ ]:
spearmans_normal

In [ ]:
spearmans_vMF

In [ ]:
ab_labels = IO.load_ome_labels('../data/integrated/antibody/NIH_F1.ome.tif')[1:]
ab_labels = [label.split('-')[-1] for label in ab_labels]

In [ ]:
ab_labels = IO.load_ome_labels('../data/integrated/antibody/NIH_F1.ome.tif')[1:]
ab_labels = [label.split('-')[-1] for label in ab_labels]
n_samples = len(val_ids)
n_abs_chans = 4

# Ordering: flatten chan (dim0) x sample (dim1)

ab_corr_df = pd.DataFrame({
    'Spearman': list(spearmans_normal.flatten(order='F')) + \
                list(spearmans_vMF.flatten(order='F')),
    'Label': np.tile(np.repeat(ab_labels, n_samples), 2),
    'Prior': ['Normal']*n_samples*n_abs_chans + ['vMF']*n_samples*n_abs_chans
})

fig, ax = plt.subplots(figsize=(7, 4))
ax = sns.barplot(x='Label', y='Spearman', data=ab_corr_df, hue='Prior')
ax.spines[['right', 'top']].set_visible(False)
sns.move_legend(ax, "upper right", bbox_to_anchor=(1.2, 0.7))
ax.set_ylabel(r'$\vert Spearman \vert$')
ax.set_xticklabels(ab_labels, rotation=45)
ax.set_title('Pseudotime vs. Ab-staining', fontsize=15)
plt.show()

In [ ]:
# Visualize ab-staining channels, what's wrong w/ GS??
ab_labels = IO.load_ome_labels('../data/integrated/antibody/NIH_F5.ome.tif')[1:]
adata_ab = IO.load_ab_stain('../data/integrated/antibody/NIH_F5.ome.tif', adata_ref=adatas_val[1])
sq.pl.spatial_scatter(adata_ab, color=ab_labels,
                      ncols=2, size=15, img=False, cmap='Purples')

---

single-sample analysis:

In [ ]:
# Compute pseudotime along the principal curve
# Comparison btw interpolation polynomials

dist_metric = 'euclidean' if model_configs.prior == 'normal' else 'geodesic'

for k in [0, 1, 2, 3]:
    print('Interpolation: k={}'.format(k))
    trajectory.compute_trajectory(adata_qz, dist_metric=dist_metric, k=k)
    adata_val.obs['t'] = adata_qz.obs['t'].values
    adata_val.obs['t_discrete'] = adata_qz.obs['t_discrete'].values
    
    sq.pl.spatial_scatter(adata_val, color='t', 
                          cmap='RdBu_r', size=20, img=False,
                          title='Pseudotim (k={})'.format(k))

In [ ]:
dist_metric = 'euclidean' if model_configs.prior == 'normal' else 'geodesic'
trajectory.compute_trajectory(adata_qz, dist_metric=dist_metric, k=0)
adata_val.obs['t'] = adata_qz.obs['t'].values
adata_val.obs['t_discrete'] = adata_qz.obs['t_discrete'].values

---